In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
# 01.ipynb와 동일한 import

load_dotenv(override=True)
# .env 파일에서 환경변수 읽어옴

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise EnvironmentError('openai api key .....')
# API 키 없으면 즉시 에러 발생

class OpenAILLM:
    def __init__(self, model: str = 'gpt-4o-mini'):
        self.client = OpenAI(api_key=api_key)
        self.model = model

    def generate(self, prompt: str) -> str:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": "너는 사용자의 질문에 친절하고 정확하게 답변하는 시스템 입니다., Return only valid JSON"
                    # 01과 차이점 1 :
                    # 01 : "You are an ecommerce recommendation assistant"
                    #      → 이커머스 전용 역할
                    # 02 : "친절하고 정확하게 답변하는 시스템"
                    #      → 뉴스/주식/날씨 등 다양한 질문을 처리해야 하므로
                    #        더 범용적인 역할로 변경
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            # 02에서는 response_format={'type':'json_object'} 가 없음
            # 01과 차이점 2 :
            # 01 : response_format 강제 → 항상 JSON만 반환
            # 02 : response_format 없음 → 자유로운 형식으로 반환 가능
            #      routerLLM()에서는 JSON이 필요하지만
            #      final_llm()에서는 자연스러운 문장 답변이 필요하므로
            #      response_format을 클래스에서 제거하고
            #      프롬프트 안에서 JSON 형식을 직접 지시하는 방식으로 변경
        )
        return response.choices[0].message.content
        # LLM 응답 텍스트 반환

llm = OpenAILLM()

In [3]:
import json
from textwrap import dedent

query = '어제 삼성전자 주식종가를 조회하고 해당 종가와 비슷한종목을 뉴스에서 검색해서 요약하고 오늘날씨에 맞는 외출복을 추천하고 데이트 장소 추천해'
# 테스트용 복합 질문
# 01의 키워드 라우터로는 절대 처리 불가능한 질문
# 주식 + 뉴스 + 날씨 + 추천 4가지 의도가 섞여있음

def routerLLM(query):
    prompt = dedent(f"""
    사용자의 질문 의도를 분석하세요.

    질문에 포함된 의도가 여러 개이면
    반드시 모든 의도를 각각 분리하여 출력하세요.
    # ↑ LLM에게 복합 질문을 분리해서 처리하라고 지시

    예를 들어:
    - 주식 + 뉴스 + 날씨
    - 뉴스 + 검색
    - 계산 + 주식
    # ↑ 복합 질문 예시를 보여줘서 LLM이 패턴을 이해하게 함
    # 이런 예시를 프롬프트에 넣는 걸 few-shot prompting 이라고 함

    처럼 복합 질문이면
    반드시 여러 개의 JSON 객체를 배열로 출력해야 합니다.

    question_type 별로 tool_query를 생성하세요.
    tool_query는 반드시 키워드 중심으로 생성하세요
    llm에 전달하는 문구가 아님을 명심해서 추론을 하지말고
    검색용 키워드로 매칭해주세요
    # ↑ tool_query가 뭔지 명확하게 지시
    # 예: "삼성전자 주식 종가를 알려주세요" (X) → LLM용 문장
    #     "삼성전자"                          (O) → 검색용 키워드

    뉴스는 api검색할수 있는 키워드중심으로,
    주식은 종목명만 추출하세요
    날씨도 검색용 키워드로 추출하세요

    지원 가능한 question_type 예시:
    - 날씨
    - 뉴스
    - 주식
    - 검색
    - 계산
    - 지도
    # ↑ LLM이 선택할 수 있는 question_type 목록을 명시
    # 이 목록 안에서만 선택하도록 유도

    [중요 규칙]
    - 질문에 포함된 모든 의도를 누락 없이 출력
    - 반드시 JSON 배열(Array) 형식으로 출력
    - 하나만 출력 금지
    - 설명문 금지
    - markdown 금지
    - ```json 금지
    # ↑ response_format을 제거했기 때문에
    #   프롬프트 안에서 직접 출력 형식을 강하게 지시
    #   ```json 금지 : LLM이 마크다운 코드블록으로 감싸면
    #                  json.loads()로 파싱할 때 에러가 나므로 금지

    [예시 입력]
    어제 삼성전자 종가 알려주고 관련 뉴스와 오늘 날씨도 알려줘

    [예시 출력]
    [
        {{
            "question_type": "주식",
            "tool_query": "삼성전자",
            "reason": "삼성전자 종가 요청"
        }},
        {{
            "question_type": "뉴스",
            "tool_query": "삼성전자 관련 최근 뉴스 검색",
            "reason": "관련 뉴스 요청"
        }},
        {{
            "question_type": "날씨",
            "tool_query": "오늘 날씨 조회",
            "reason": "날씨 요청"
        }}
    ]
    # ↑ 입력/출력 예시를 직접 보여주는 것 = few-shot prompting
    # LLM이 형식을 정확히 따라하게 만드는 가장 효과적인 방법
    # {{ }} : f-string 안에서 중괄호를 문자 그대로 출력할 때 쓰는 이스케이프

    [질문]
    {query}
    # ↑ 실제 사용자 질문이 여기 삽입됨

    [출력]
    반드시 유효한 JSON 배열만 출력하세요.
    """)

    result = llm.generate(prompt)
    # LLM에게 프롬프트를 보내서 JSON 배열 문자열을 받아옴
    # 예: '[{"question_type":"주식","tool_query":"삼성전자",...},...]'

    json_result = json.loads(result)
    # JSON 배열 문자열 → 파이썬 리스트로 변환
    # '[{...},{...}]' → [{...}, {...}]
    # 이후 for문으로 각 의도를 순회할 수 있게 됨

    return json_result
    # 의도가 분리된 딕셔너리들의 리스트 반환


# 실행 테스트
router_result = routerLLM(query)
for item in router_result:
    print(item)

{'question_type': '주식', 'tool_query': '삼성전자', 'reason': '삼성전자 주식종가 조회 요청'}
{'question_type': '뉴스', 'tool_query': '삼성전자 주식종가 비슷한 종목', 'reason': '해당 종가와 비슷한 종목 뉴스 검색 요청'}
{'question_type': '날씨', 'tool_query': '오늘 날씨', 'reason': '오늘 날씨에 맞는 외출복 추천 요청'}
{'question_type': '검색', 'tool_query': '데이트 장소 추천', 'reason': '데이트 장소 추천 요청'}


In [7]:
# 네이버 뉴스 검색 도구 (01.ipynb와 동일)
import os
import re
import json
import html
import urllib.request
from datetime import datetime
from dotenv import load_dotenv
load_dotenv(override=True)

def _format_date(pubdate):
    return datetime.strptime(pubdate, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d")
    # "Mon, 27 May 2026 10:00:00 +0900" → "2026-05-27"

def _format_str(text):
    return html.unescape(re.sub(r'<[^>]+>', "", text))
    # HTML 태그 제거 + 특수문자 복원

client_id = os.getenv('NAVER_CLIENT_ID')
client_secret = os.getenv('NAVER_CLIENT_SECRET')

items = []
def search_naver_news(query: str, display: int = 3) -> list[dict]:
    encText = urllib.parse.quote(query)
    encText += f'&display={display}&sort=date'
    url = "https://openapi.naver.com/v1/search/news?query=" + encText
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id", client_id)
    request.add_header("X-Naver-Client-Secret", client_secret)
    response = urllib.request.urlopen(request)
    rescode = response.getcode()
    if rescode == 200:
        response_body = response.read().decode('utf-8')
        result = json.loads(response_body)
        for row in result.get('items'):
            items.append({
                'title':   _format_str(row.get('title')),
                'content': _format_str(row.get('description')),
                'date':    _format_date(row.get('pubDate')),
                'link':    row.get('link')
            })
    return items
# 01.ipynb와 완전히 동일한 네이버 뉴스 검색 함수
# 자세한 주석은 01.ipynb 5단계 참고


# -------------------------------------------------------
# 핵심 : 도구 실행 함수
# -------------------------------------------------------
tool_results = []
# 각 도구의 실행 결과를 누적해서 담을 리스트
# 01.ipynb는 도구 하나만 실행하고 바로 반환했지만
# 02.ipynb는 여러 도구를 실행하고 결과를 전부 여기에 누적함
# 나중에 final_llm()에게 이 리스트 전체를 넘겨서 종합 답변 생성

def excute_tools(router_result):
    # router_result : 2단계 routerLLM()이 반환한 딕셔너리 리스트
    # 예: [{"question_type":"뉴스", "tool_query":"삼성전자"},
    #      {"question_type":"주식", "tool_query":"삼성전자"}]

    for row in router_result:
        # 리스트를 순회하면서 각 의도를 하나씩 처리
        
        query_type = row.get('question_type')
        # 현재 처리할 의도의 타입을 꺼냄
        # 예: "뉴스", "주식", "날씨"

        print(f'tool : {query_type}')
        # 어떤 도구가 실행되는지 확인용 출력

        if query_type == '뉴스':
            query = row.get('tool_query')
            # routerLLM이 추출한 검색 키워드를 꺼냄
            # 예: "삼성전자 관련 최근 뉴스"

            news_result = search_naver_news(row.get('tool_query'))
            # 네이버 뉴스 API로 검색 실행

            tool_results.append({
                'question_type': '뉴스',
                'query': query,
                'result': news_result
                # 도구 실행 결과를 tool_results 리스트에 추가
                # 01.ipynb는 바로 return 했지만
                # 02.ipynb는 append로 누적 → 여러 도구 결과를 한번에 모음
            })

        # other tools
        # ↑ 주식, 날씨, 지도 등 다른 도구들은 아직 미구현
        # 현재는 뉴스 도구만 실제로 동작함
        # 나머지는 추후 동일한 패턴으로 추가하면 됨
        # if query_type == '주식':
        #     stock_result = search_stock(row.get('tool_query'))
        #     tool_results.append({...})

    return tool_results
    # 모든 도구 실행 결과가 누적된 리스트 반환
    # 예: [
    #       {"question_type":"뉴스", "query":"삼성전자", "result":[{뉴스1},{뉴스2}]},
    #       {"question_type":"뉴스", "query":"날씨",     "result":[{뉴스1},{뉴스2}]}
    #     ]


# 실행 테스트
router_result = routerLLM(query)
# 2단계에서 만든 routerLLM으로 의도 분석

tool_results = excute_tools(router_result)
# 분석된 의도에 맞는 도구 실행 후 결과 수집

print(json.dumps(tool_results, ensure_ascii=False, indent=2))
# 수집된 결과 확인

tool : 주식
tool : 뉴스
tool : 날씨
tool : 검색
[
  {
    "question_type": "뉴스",
    "query": "삼성전자 주식종가 비슷한 종목",
    "result": [
      {
        "title": "[이제 1만 간다]③코스피 재평가는 현재진행형...\"8000 넘어 1만피 가능...",
        "content": "KB증권도 반도체 투톱인 삼성전자와 SK하이닉스의 영업이익 성장세를 기반으로 국내 증시의 경쟁력이... 일각에서는 2021년과 비슷한 수급 구도가 전개될 수 있다는 우려로 인해 '개미 설거지 장세'라는 이야기도... ",
        "date": "2026-05-28",
        "link": "http://www.metroseoul.co.kr/article/20260528500056"
      },
      {
        "title": "[기자의 눈] '팔천피' 축포의 이면엔 \"주린이도 돈 번다\"의 리스크",
        "content": "이날 상장한 삼성전자·SK하이닉스 단일 종목 레버리지에 투자자 시선이 쏠리는 것이 당연하다. 유가증권... 사실 우리는 비슷한 장면을 최근에도 경험한 적이 있다. 코스피는 2020년 3월 팬데믹 충격으로 장중 1439.43까지... ",
        "date": "2026-05-27",
        "link": "https://www.ajunews.com/view/20260527141533308"
      },
      {
        "title": "'삼성전자 -18%' … 혼란 야기한 NXT \"변동성 장치 추가\"",
        "content": "삼성전자는 SK하이닉스와 함께 국내 대표 반도체주로 꼽히는 만큼 시장 전체 투자심리에 미치는 영향도 큰 종목이다. 온라인 주식 커뮤니티에는 \"지난 2월에도 비슷한 일이 있었는데 또 발생했다\", \"오류 체결인지... ",
        "da

In [8]:
# -------------------------------------------------------
# 메시지 조립 함수
# -------------------------------------------------------
def make_message(user_query: str, tool_results: list[dict]):
    prompt = f'''
    너는 다양한 도구의 결과를 종합해서 사용자 질문에 답변하는 ai assistant 입니다.

    [사용자질문]
    {user_query}
    # ↑ 원래 사용자가 했던 질문 전체가 여기 삽입됨

    [도구실행결과]
    {json.dumps(tool_results, ensure_ascii=False, indent=2)}
    # ↑ 3단계에서 수집한 모든 도구 결과가 여기 통째로 삽입됨
    # json.dumps()로 보기 좋게 변환해서 LLM이 읽기 쉽게 만듦
    # ensure_ascii=False : 한글 깨짐 방지
    # indent=2 : 들여쓰기로 구조를 명확하게

    [지침]
    - tool 결과를 기반으로 답변하세요
    - 필요한 경우 여러 tool 결과를 종합하세요
    - 지도 주식 날씨 검색 추천등 다양한 데이터가 포함될수 있습니다.
    - tool 결과내에 있는 내용에 우선순위를 높게해서 해당 데이터기반으로 답변하세요
    # ↑ LLM이 자기 지식이 아닌 tool 결과를 우선으로 답변하도록 지시
    # 이게 없으면 LLM이 tool 결과를 무시하고 학습 데이터로만 답변할 수 있음
    '''

    return [
        {
            "role": "system",
            "content": "너는 여러 tool결과를 조합해서 최종 답변을 생성하는 ai agent 입니다."
            # system 메시지로 LLM의 역할을 최종 답변 생성자로 설정
        },
        {
            "role": "user",
            "content": prompt
            # user 메시지로 사용자 질문 + 도구 결과 전달
        }
    ]
    # ↑ 메시지 리스트를 반환
    # generate()에서 바로 쓰던 messages 형식과 동일
    # 별도 함수로 분리한 이유 :
    # final_llm_openai()와 final_llm_qween() 두 곳에서
    # 동일한 메시지를 재사용하기 위해


# -------------------------------------------------------
# OpenAI 최종 답변 생성
# -------------------------------------------------------
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
# OpenAI 클라이언트 객체 생성
# llm 객체와 별도로 만드는 이유 :
# llm 객체는 라우터용 (temperature=0, JSON 출력)
# 여기선 최종 답변용 (temperature=0.3, 자연스러운 문장 출력)
# 용도가 다르므로 분리

def final_llm_openai(user_query: str, tool_results: list[dict]):
    result = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=make_message(user_query, tool_results),
        # make_message()로 조립한 메시지 리스트를 그대로 넘김
        temperature=0.3
        # 0.3 : 라우터(0)보다 약간 높게 설정
        # 최종 답변은 자연스러운 문장이 필요하므로
        # 완전히 고정(0)보다 약간의 유연성을 줌
    )
    return result.choices[0].message.content
    # 최종 답변 텍스트 반환


# -------------------------------------------------------
# HuggingFace 모델 설정 (오픈소스 모델로 교체)
# -------------------------------------------------------
from huggingface_hub import notebook_login
# HuggingFace에 로그인하기 위한 라이브러리
# notebook_login() : 주피터 노트북에서 HuggingFace 토큰을 입력받는 UI를 띄움

notebook_login()
# HuggingFace 계정의 토큰을 입력해서 로그인
# 토큰은 huggingface.co → 계정설정 → Access Tokens에서 발급

from huggingface_hub import get_token, whoami
HF_TOKEN = get_token()
# 로그인 후 저장된 토큰을 가져옴

if not HF_TOKEN:
    raise RuntimeError('Hugging Face 토큰이 저장되지 않았습니다. 로그인 셀을 다시 실행하세요.')

info = whoami(token=HF_TOKEN)
# 토큰으로 내 계정 정보를 가져와서 로그인이 정상인지 확인

print('logged in user:', info.get('name'))
print('token prefix:', HF_TOKEN[:6] + '***')
# 앞 6자리만 출력해서 토큰이 노출되지 않게 확인


# -------------------------------------------------------
# HuggingFace 모델 로드
# -------------------------------------------------------
from transformers import AutoModelForCausalLM, AutoTokenizer
# transformers : HuggingFace의 핵심 라이브러리
# AutoModelForCausalLM : 텍스트 생성 모델을 자동으로 로드
# AutoTokenizer : 해당 모델에 맞는 토크나이저를 자동으로 로드
# 토크나이저 : 문자열을 모델이 이해할 수 있는 숫자(토큰)로 변환하는 도구

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
# 사용할 모델 이름
# Qwen2.5-0.5B : 파라미터 수가 0.5B(5억개)인 경량 모델
# Instruct : 지시문을 따르도록 파인튜닝된 버전
# 0.5B라서 무겁지 않아 로컬에서도 실행 가능

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    # 모델 가중치의 데이터 타입을 자동으로 선택
    # GPU가 있으면 float16, CPU면 float32로 자동 설정
    device_map="auto"
    # 모델을 어떤 장치에 올릴지 자동으로 결정
    # GPU가 있으면 GPU, 없으면 CPU에 올림
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
# 모델에 맞는 토크나이저 로드


# -------------------------------------------------------
# HuggingFace 최종 답변 생성
# -------------------------------------------------------
def final_llm_qween(user_query: str, tool_results: list[dict]):

    text = tokenizer.apply_chat_template(
        make_message(user_query, tool_results),
        # make_message()로 조립한 메시지 리스트를 넘김
        # OpenAI는 messages 형식을 알아서 처리하지만
        # HuggingFace 모델은 모델마다 고유한 입력 형식이 있음
        # apply_chat_template()이 messages를 해당 모델 형식으로 변환해줌
        tokenize=False,
        # 텍스트 변환만 하고 아직 토큰으로 변환하지 않음
        # True면 숫자 배열로 바로 변환되는데
        # 여기선 일단 문자열로 받고 다음 줄에서 따로 토큰화함
        add_generation_prompt=True
        # 모델이 답변을 생성하도록 유도하는 특수 토큰을 끝에 추가
        # 이게 없으면 모델이 답변을 생성하지 않고 멈출 수 있음
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    # tokenizer() : 문자열을 토큰 숫자 배열로 변환
    # [text] : 리스트로 감싸는 이유 → 배치(batch) 처리를 위해
    #          모델은 여러 입력을 한번에 처리할 수 있는데
    #          1개여도 리스트 형태로 넘겨야 함
    # return_tensors="pt" : 파이토치 텐서 형식으로 반환
    # .to(model.device) : 모델이 있는 장치(GPU/CPU)로 데이터를 이동
    #                     모델과 입력 데이터가 같은 장치에 있어야 연산 가능

    generated_ids = model.generate(
        **model_inputs,
        # **model_inputs : 딕셔너리를 언패킹해서 키워드 인자로 전달
        # model_inputs 안에 input_ids, attention_mask 등이 들어있음
        max_new_tokens=512
        # 새로 생성할 토큰의 최대 개수
        # 너무 크면 느려지고 너무 작으면 답변이 잘림
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    # 입력으로 넣은 토큰을 제거하고 새로 생성된 토큰만 추출
    # model.generate()는 입력 토큰 + 생성 토큰을 합쳐서 반환하므로
    # 입력 길이만큼 앞을 잘라내야 순수하게 생성된 답변만 남음
    # zip() : input_ids와 generated_ids를 쌍으로 묶어서 순회
    # output_ids[len(input_ids):] : 생성된 전체에서 입력 길이 이후만 슬라이싱

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    # batch_decode() : 토큰 숫자 배열 → 문자열로 변환 (토큰화의 반대 과정)
    # skip_special_tokens=True : <|im_start|> 같은 특수 토큰을 제거하고 변환
    # [0] : 배치의 첫 번째(유일한) 결과를 꺼냄

    return response
    # 최종 답변 문자열 반환


# -------------------------------------------------------
# 파이프라인 전체 실행
# -------------------------------------------------------
query = '붕괴사고에 대해서'

router_result = routerLLM(query)
# 2단계 : LLM이 질문 의도 분석
# → [{"question_type":"뉴스", "tool_query":"붕괴사고"}]

tool_results = excute_tools(router_result)
# 3단계 : 분석된 의도에 맞는 도구 실행
# → [{"question_type":"뉴스", "query":"붕괴사고", "result":[{뉴스1},{뉴스2}]}]

final_result = final_llm_openai(query, tool_results)
# 4단계 : 도구 결과를 종합해서 최종 답변 생성

print(final_result)

RuntimeError: Hugging Face 토큰이 저장되지 않았습니다. 로그인 셀을 다시 실행하세요.